In [1]:
from sqlflex import parse

sql_input = """
SELECT TOP 10 *
FROM Sales
WHERE (tot / 2) !< 8 AND year < 2025
OPTION (FAST 10) ;
"""

ast,stats,_ = parse(sql_input, dialect='TSQL')

[sqlflex] Parsing SQL (TSQL dialect)...
[sqlflex] LLM enabled — segmenter: gpt-4.1, validator: gpt-4.1
[sqlflex] Grammar parse failed for 'root', falling back to LLM segmentation...
[sqlflex] Grammar parse failed for 'selectClause', falling back to LLM segmentation...
[sqlflex] Grammar parse failed for 'whereClause', falling back to LLM segmentation...
[sqlflex] Parsing complete in 5.43s — LLM calls: 5, tokens: 3561↑ 146↓


In [2]:
print(stats) # LLM-related stats
ast.pretty_print() # Pretty print the AST

{'called_times': 5, 'expr_called_times': 2, 'fix_called_times': 0, 'fails': 0, 'expr_cost': 0.0, 'total_cost': 0.0, 'total_time': 5.429319858551025, 'prompt_tokens': 3561, 'completion_tokens': 146}
SelectStmt(
  others=[
    Other(
      value=OPTION (FAST 10)
      pos=3)]
  select=Select(
    projections=[
      Projection(
        column=Column(
          name=*))]
    selectSpecifiers=TOP 10)
  from_=From(
    tables=[
      TableReference(
        table=Table(
          name=Sales))])
  where=Where(
    expr=Operation(
      op=AND
      left=Operation(
        op=!<
        left=Operation(
          hasParentheses=True
          op=/
          left=Identifier(
            value=tot)
          right=Number(
            value=2))
        right=Number(
          value=8))
      right=Operation(
        op=<
        left=Literal(
          value=year)
        right=Number(
          value=2025)))))


In [3]:
from sqlflex import pretty_print
pretty_print(ast)

'SELECT TOP 10 * FROM Sales WHERE (tot / 2) !< 8 AND year < 2025 OPTION (FAST 10);'

In [4]:
# Suppose we want to detect an anti-pattern: Use <= instead of !<
from sqlflex.sqlast.ASTNodes import Operation
from sqlflex import findall, transform, pretty_print
not_less_than: list[Operation] = findall('Operation', ast, filter_func=lambda x: x.op == '!<')
for expr in not_less_than:
  print(f"Use <= instead of !< in expression: {pretty_print(expr)}")
  
# Fix: Use <= instead of !<
def fix_not_less_than(node):
    if isinstance(node, Operation) and node.op == '!<':
        node.op = '<='
fixed_ast = transform(ast, fix_not_less_than, inplace=False)
print(f"Fixed query: {pretty_print(fixed_ast)}")

Use <= instead of !< in expression: (tot / 2) !< 8;
Fixed query: SELECT TOP 10 * FROM Sales WHERE (tot / 2) <= 8 AND year < 2025 OPTION (FAST 10);
